# 06 · Modelado predictivo

Este notebook ejecuta el entrenamiento y evaluación base de los modelos predictivos definidos para el TFM **Evaluación del valor incremental de parámetros ECG en riesgo cardiometabólico**.

El proceso utiliza los datasets experimentales construidos en `05_construccion_endpoint.ipynb` y genera métricas por escenario, modelo y subconjunto estructural cuando la información de batería se encuentra disponible.

Responsabilidad del notebook:

- entrenar modelos supervisados por escenario experimental;
- calcular métricas globales por escenario y modelo;
- conservar predicciones individuales con trazabilidad analítica;
- calcular métricas por subconjunto estructural a partir de las predicciones de evaluación;
- guardar modelos entrenados y artefactos reproducibles.

La interpretación incremental entre escenarios corresponde al notebook `07_evaluacion_incremental_ecg.ipynb`.

## 1. Entradas y salidas

### Entradas esperadas

Archivos generados por `05_construccion_endpoint.ipynb`:

```text
Dataset_E1_Clinico_TFM.xlsx
Dataset_E2_Clinico_NLP_TFM.xlsx
Dataset_E3_Clinico_ECG_TFM.xlsx
Dataset_E4_Clinico_NLP_ECG_TFM.xlsx
```

### Salidas generadas

```text
Metricas_Modelado_TFM.xlsx
Metricas_Modelado_Subsets_TFM.xlsx
Predicciones_Modelos_TFM.xlsx
Variables_Modelado_TFM.xlsx
Reporte_Modelado_Predictivo_TFM.txt
modelos_entrenados/
```

`Metricas_Modelado_TFM.xlsx` contiene dos hojas principales:

```text
METRICAS
METRICAS_POR_BATERIA
```

La hoja `METRICAS` resume el desempeño global por escenario y modelo. La hoja `METRICAS_POR_BATERIA` resume el desempeño por subconjunto estructural cuando existe una columna compatible con batería en los datasets de entrada.

In [9]:
# Instalación de dependencias requeridas.
# Ejecutar esta celda si el entorno local no tiene alguna dependencia instalada.

import importlib.util
import subprocess
import sys

DEPENDENCIAS = {
    "pandas": "pandas",
    "numpy": "numpy",
    "openpyxl": "openpyxl",
    "xlsxwriter": "XlsxWriter",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm",
}

for modulo, paquete in DEPENDENCIAS.items():
    if importlib.util.find_spec(modulo) is None:
        print(f"Instalando dependencia faltante: {paquete}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", paquete])
        except Exception as exc:
            print(f"No fue posible instalar {paquete}: {exc}")
    else:
        print(f"Dependencia disponible: {modulo}")


Dependencia disponible: pandas
Dependencia disponible: numpy
Dependencia disponible: openpyxl
Dependencia disponible: xlsxwriter
Dependencia disponible: sklearn
Dependencia disponible: joblib
Dependencia disponible: xgboost
Dependencia disponible: lightgbm


In [10]:
import os
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "RIESGO_CARDIOMETABOLICO"

DATA_DIR = Path(".")
OUTPUT_DIR = Path(".")
MODEL_DIR = OUTPUT_DIR / "modelos_entrenados"
MODEL_DIR.mkdir(exist_ok=True)

SCENARIO_FILES = {
    "E1_CLINICO": "Dataset_E1_Clinico_TFM.xlsx",
    "E2_CLINICO_NLP": "Dataset_E2_Clinico_NLP_TFM.xlsx",
    "E3_CLINICO_ECG": "Dataset_E3_Clinico_ECG_TFM.xlsx",
    "E4_CLINICO_NLP_ECG": "Dataset_E4_Clinico_NLP_ECG_TFM.xlsx",
}

# Nombres canónicos alineados con el TFM y con el notebook 07.
MODEL_DISPLAY_NAMES = {
    "LogisticRegression": "Logistic Regression",
    "RandomForest": "Random Forest",
    "XGBoost": "XGBoost",
    "LightGBM": "LightGBM",
}

TRACE_COLS = [
    # Identificadores y trazabilidad de paciente/registro
    "PACIENTE_ID", "REGISTRO_ID",

    # Trazabilidad estructural de pseudo-baterías.
    # Estas columnas se conservan para auditoría y métricas por subconjunto,
    # pero nunca deben entrar como predictores del modelo.
    "SUBSET_BATERIA", "SUBSET_EXAMENES", "BATERIA_CLUSTER",
    "BATERIA", "BATERIA_ESTRUCTURAL", "CLUSTER_BATERIA",
    "subset_bateria", "subset_examenes", "bateria_cluster", "bateria", "subset",

    # Trazabilidad privada o administrativa
    "clave_matching", "archivo_origen", "ruta_relativa", "nombre_paciente", "nombre_paciente_norm",
    "fecha", "fecha_examen", "fecha_atencion_norm", "clave_hash_privada", "clave_hash_ecg",
]

BATTERY_COL_CANDIDATES = [
    "SUBSET_BATERIA", "SUBSET_EXAMENES", "BATERIA", "BATERIA_ESTRUCTURAL",
    "BATERIA_CLUSTER", "CLUSTER_BATERIA", "subset_bateria", "subset_examenes",
    "bateria", "bateria_cluster", "subset"
]

ENDPOINT_FACTORS = [
    "FLAG_OBESIDAD_IMC",
    "FLAG_PA_SISTOLICA_ALTA",
    "FLAG_PA_DIASTOLICA_ALTA",
    "FLAG_GLICEMIA_ALTA",
    "FLAG_LDL_ALTO",
    "FLAG_TRIGLICERIDOS_ALTOS",
    "FLAG_HDL_BAJO",
    "Diabetes_bin",
    "ANT_HTA",
    "ANT_DIABETES",
    "ANT_DISLIPIDEMIA",
    "ANT_OBESIDAD",
]

OTHER_EXCLUDED = [
    TARGET,
    "INDICE_RIESGO_CARDIOMETABOLICO",
    "num_factores_endpoint_disponibles",
    "endpoint_valido",
]

METRICS_PATH = OUTPUT_DIR / "Metricas_Modelado_TFM.xlsx"
METRICS_SUBSETS_PATH = OUTPUT_DIR / "Metricas_Modelado_Subsets_TFM.xlsx"
PREDICTIONS_PATH = OUTPUT_DIR / "Predicciones_Modelos_TFM.xlsx"
FEATURES_PATH = OUTPUT_DIR / "Variables_Modelado_TFM.xlsx"
REPORT_PATH = OUTPUT_DIR / "Reporte_Modelado_Predictivo_TFM.txt"


## 2. Funciones auxiliares

In [11]:
def load_scenarios(data_dir: Path, scenario_files: dict) -> dict:
    datasets = {}
    missing = []
    for scenario, filename in scenario_files.items():
        path = data_dir / filename
        if not path.exists():
            missing.append(str(path))
            continue
        datasets[scenario] = pd.read_excel(path)
    if missing:
        raise FileNotFoundError(
            "No se encontraron los siguientes datasets de entrada:\n" + "\n".join(missing)
        )
    return datasets


def validate_required_columns(df: pd.DataFrame, scenario: str) -> None:
    required = ["PACIENTE_ID", "REGISTRO_ID", TARGET]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{scenario}: faltan columnas requeridas: {missing}")
    if df[TARGET].isna().any():
        raise ValueError(f"{scenario}: la variable objetivo contiene valores nulos")
    unique_values = sorted(pd.Series(df[TARGET]).dropna().unique().tolist())
    if set(unique_values) - {0, 1}:
        raise ValueError(f"{scenario}: {TARGET} debe ser binaria 0/1. Valores detectados: {unique_values}")
    if df[TARGET].nunique() < 2:
        raise ValueError(f"{scenario}: {TARGET} contiene una sola clase; no permite entrenamiento supervisado")


def validate_endpoint_leakage(df: pd.DataFrame, scenario: str) -> list:
    present = [c for c in ENDPOINT_FACTORS if c in df.columns]
    if present:
        print(f"{scenario}: columnas de endpoint presentes en el archivo y serán excluidas del entrenamiento: {present}")
    return present


def get_feature_columns(df: pd.DataFrame) -> list:
    excluded = set(TRACE_COLS + ENDPOINT_FACTORS + OTHER_EXCLUDED)

    # Exclusión defensiva por nombre normalizado para impedir que columnas
    # estructurales o de trazabilidad entren al preprocesador aunque cambie
    # su capitalización o variante de nomenclatura.
    excluded_norm = {str(c).strip().lower() for c in excluded}

    features = []
    for col in df.columns:
        col_norm = str(col).strip().lower()
        if str(col).startswith("Unnamed"):
            continue
        if col in excluded or col_norm in excluded_norm:
            continue
        features.append(col)

    if not features:
        raise ValueError("No quedan variables predictoras después de aplicar exclusiones")
    return features


def split_feature_types(df: pd.DataFrame, feature_cols: list) -> tuple:
    X = df[feature_cols].copy()
    numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = [c for c in feature_cols if c not in numeric_cols]
    return numeric_cols, categorical_cols


def find_battery_column(df: pd.DataFrame) -> str | None:
    for col in BATTERY_COL_CANDIDATES:
        if col in df.columns:
            return col
    return None


def make_preprocessor(numeric_cols: list, categorical_cols: list, scale_numeric: bool) -> ColumnTransformer:
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_transformer = Pipeline(steps=numeric_steps)

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    transformers = []
    if numeric_cols:
        transformers.append(("num", numeric_transformer, numeric_cols))
    if categorical_cols:
        transformers.append(("cat", categorical_transformer, categorical_cols))

    return ColumnTransformer(transformers=transformers, remainder="drop")


def load_optional_models() -> dict:
    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            solver="lbfgs",
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
        ),
    }

    try:
        from xgboost import XGBClassifier
        models["XGBoost"] = XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    except Exception as exc:
        print(f"XGBoost no disponible. Se omite modelo. Detalle: {exc}")

    try:
        from lightgbm import LGBMClassifier
        models["LightGBM"] = LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
            verbose=-1,
        )
    except Exception as exc:
        print(f"LightGBM no disponible. Se omite modelo. Detalle: {exc}")

    return models



def extract_numeric_scalar(value):
    """Convierte valores escalares, arrays de un elemento o strings tipo '[1.23E-1]' a float."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    if isinstance(value, (list, tuple, np.ndarray, pd.Series)):
        arr = np.asarray(value).ravel()
        if arr.size == 0:
            return np.nan
        return extract_numeric_scalar(arr[0])
    if isinstance(value, str):
        s = value.strip()
        # Elimina corchetes residuales generados por serialización de arrays de un elemento.
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1].strip()
        s = s.replace(",", ".")
        return float(s)
    return float(value)


def to_numeric_1d(values, name: str = "score") -> np.ndarray:
    """Normaliza vectores de predicción a ndarray 1D de floats escalares."""
    if isinstance(values, pd.Series):
        raw = values.to_numpy()
    else:
        raw = np.asarray(values)

    if raw.ndim == 2:
        if raw.shape[1] >= 2:
            raw = raw[:, 1]
        elif raw.shape[1] == 1:
            raw = raw[:, 0]
    raw = np.asarray(raw, dtype=object).reshape(-1)
    out = np.array([extract_numeric_scalar(v) for v in raw], dtype=float)
    if np.isnan(out).any():
        raise ValueError(f"{name}: contiene valores no convertibles a float")
    return out


def predict_scores(model: Pipeline, X_test: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return to_numeric_1d(model.predict_proba(X_test), name="predict_proba")
    if hasattr(model, "decision_function"):
        score = to_numeric_1d(model.decision_function(X_test), name="decision_function")
        return (score - score.min()) / (score.max() - score.min() + 1e-12)
    return to_numeric_1d(model.predict(X_test), name="predict")


def safe_auc(metric_fn, y_true, y_score):
    y_true = pd.Series(y_true)
    if y_true.nunique(dropna=True) < 2:
        return np.nan
    try:
        return metric_fn(y_true, y_score)
    except Exception:
        return np.nan


def evaluate_predictions(y_true, y_pred, y_score) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": safe_auc(roc_auc_score, y_true, y_score),
        "AUPRC": safe_auc(average_precision_score, y_true, y_score),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }
    return metrics


def evaluate_predictions_by_battery(pred_df: pd.DataFrame, scenario: str, model_name: str, battery_col: str | None) -> list:
    if battery_col is None or battery_col not in pred_df.columns:
        return []

    rows = []
    for battery_value, group in pred_df.groupby(battery_col, dropna=False):
        y_true = group["y_real"].astype(int)
        y_pred = group["y_pred"].astype(int)
        y_score = group["y_score"].astype(float)
        metrics = evaluate_predictions(y_true, y_pred, y_score)
        rows.append({
            "escenario": scenario,
            "modelo": model_name,
            "columna_bateria": battery_col,
            "BATERIA": battery_value,
            "n_test_bateria": int(len(group)),
            "n_positivos_bateria": int((y_true == 1).sum()),
            "n_negativos_bateria": int((y_true == 0).sum()),
            **metrics,
        })
    return rows


## 3. Carga y validación de datasets E1–E4

In [12]:
datasets = load_scenarios(DATA_DIR, SCENARIO_FILES)

summary_rows = []
for scenario, df in datasets.items():
    validate_required_columns(df, scenario)
    leakage_cols = validate_endpoint_leakage(df, scenario)
    feature_cols = get_feature_columns(df)
    target_counts = df[TARGET].value_counts(dropna=False).to_dict()
    battery_col = find_battery_column(df)
    n_batteries = int(df[battery_col].nunique(dropna=True)) if battery_col else 0
    summary_rows.append({
        "escenario": scenario,
        "registros": len(df),
        "columnas_totales": df.shape[1],
        "features_entrenamiento": len(feature_cols),
        "factores_endpoint_presentes_excluidos": len(leakage_cols),
        "columna_bateria_detectada": battery_col or "NO_DISPONIBLE",
        "n_baterias_detectadas": n_batteries,
        "clase_0": int(target_counts.get(0, 0)),
        "clase_1": int(target_counts.get(1, 0)),
        "prevalencia_clase_1": float(df[TARGET].mean()),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


,escenario,registros,columnas_totales,features_entrenamiento,factores_endpoint_presentes_excluidos,columna_bateria_detectada,n_baterias_detectadas,clase_0,clase_1,prevalencia_clase_1
0,E1_CLINICO,3779,29,22,0,SUBSET_BATERIA,4,3339,440,0.116433
1,E2_CLINICO_NLP,3779,41,34,0,SUBSET_BATERIA,4,3339,440,0.116433
2,E3_CLINICO_ECG,3779,40,33,0,SUBSET_BATERIA,4,3339,440,0.116433
3,E4_CLINICO_NLP_ECG,3779,52,45,0,SUBSET_BATERIA,4,3339,440,0.116433


## 4. Entrenamiento y evaluación por escenario

El entrenamiento se realiza a nivel de escenario completo. Las métricas globales se calculan sobre el conjunto de prueba estratificado. Cuando existe columna de batería, las mismas predicciones de prueba se agrupan por subconjunto estructural para calcular métricas adicionales por batería.

Esta estrategia mantiene una partición común por escenario y evita entrenamientos no comparables entre subconjuntos de distinto tamaño.

In [13]:
all_metrics = []
all_subset_metrics = []
all_predictions = []
all_features = []
trained_models = {}

base_models = load_optional_models()
print("Modelos disponibles:", [MODEL_DISPLAY_NAMES.get(k, k) for k in base_models.keys()])

for scenario, df in datasets.items():
    print("=" * 80)
    print(f"Escenario: {scenario}")

    feature_cols = get_feature_columns(df)
    X = df[feature_cols].copy()
    y = df[TARGET].astype(int)

    numeric_cols, categorical_cols = split_feature_types(df, feature_cols)
    battery_col = find_battery_column(df)

    stratify = y if y.nunique() == 2 else None
    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X, y, df.index,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=stratify,
    )

    scenario_model_dir = MODEL_DIR / scenario
    scenario_model_dir.mkdir(exist_ok=True)

    for model_key, estimator in base_models.items():
        model_name = MODEL_DISPLAY_NAMES.get(model_key, model_key)
        print(f"Entrenando {model_name}...")
        scale_numeric = model_key == "LogisticRegression"
        preprocessor = make_preprocessor(numeric_cols, categorical_cols, scale_numeric=scale_numeric)
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ])

        status = "OK"
        error_message = ""
        try:
            pipeline.fit(X_train, y_train)
            y_pred = to_numeric_1d(pipeline.predict(X_test), name="y_pred").astype(int)
            y_score = predict_scores(pipeline, X_test)
            y_score = to_numeric_1d(y_score, name="y_score")
            metrics = evaluate_predictions(y_test, y_pred, y_score)

            model_path = scenario_model_dir / f"{model_key}.joblib"
            joblib.dump(pipeline, model_path)
            trained_models[(scenario, model_name)] = str(model_path)

            pred_df = pd.DataFrame({
                "escenario": scenario,
                "modelo": model_name,
                "modelo_key": model_key,
                "index_original": idx_test,
                "y_real": y_test.values,
                "y_pred": y_pred,
                "y_score": y_score.astype(float),
            })
            if not pd.api.types.is_numeric_dtype(pred_df["y_score"]):
                raise TypeError("y_score debe ser numérico antes de exportar predicciones")
            # Conservar columnas de trazabilidad en predicciones sin usarlas como features.
            trace_to_keep = [
                "PACIENTE_ID", "REGISTRO_ID", "SUBSET_BATERIA", "SUBSET_EXAMENES",
                "BATERIA_CLUSTER", "BATERIA", "BATERIA_ESTRUCTURAL", "CLUSTER_BATERIA"
            ]
            if battery_col and battery_col not in trace_to_keep:
                trace_to_keep.append(battery_col)

            for trace_col in trace_to_keep:
                if trace_col in df.columns and trace_col not in pred_df.columns:
                    pred_df[trace_col] = df.loc[idx_test, trace_col].values

            all_predictions.append(pred_df)

            subset_rows = evaluate_predictions_by_battery(pred_df, scenario, model_name, battery_col)
            all_subset_metrics.extend(subset_rows)

        except Exception as exc:
            status = "ERROR"
            error_message = repr(exc)
            metrics = {
                "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
                "F1": np.nan, "ROC_AUC": np.nan, "AUPRC": np.nan,
                "TN": np.nan, "FP": np.nan, "FN": np.nan, "TP": np.nan,
            }

        metric_row = {
            "escenario": scenario,
            "modelo": model_name,
            "modelo_key": model_key,
            "estado": status,
            "error": error_message,
            "n_total": len(df),
            "n_train": len(X_train),
            "n_test": len(X_test),
            "features": len(feature_cols),
            "numeric_features": len(numeric_cols),
            "categorical_features": len(categorical_cols),
            "columna_bateria": battery_col or "NO_DISPONIBLE",
            "prevalencia_train": float(y_train.mean()),
            "prevalencia_test": float(y_test.mean()),
            **metrics,
        }
        all_metrics.append(metric_row)

    for col in feature_cols:
        modality = "CLINICA"
        if col.startswith("ANT_"):
            modality = "NLP"
        elif col.startswith("ECG_") or col in ["QT", "QRS_AXIS", "RV5", "SV1", "RV1", "SV5"]:
            modality = "ECG"
        all_features.append({
            "escenario": scenario,
            "variable": col,
            "modalidad_inferida": modality,
            "tipo_dato": str(df[col].dtype),
            "n_nulos": int(df[col].isna().sum()),
            "pct_nulos": float(df[col].isna().mean() * 100),
        })

metrics_df = pd.DataFrame(all_metrics)
subset_metrics_df = pd.DataFrame(all_subset_metrics)
predictions_df = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()
features_df = pd.DataFrame(all_features)

metrics_df.sort_values(["escenario", "ROC_AUC", "AUPRC"], ascending=[True, False, False])


Modelos disponibles: ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM']
Escenario: E1_CLINICO
Entrenando Logistic Regression...
Entrenando Random Forest...
Entrenando XGBoost...
Entrenando LightGBM...
Escenario: E2_CLINICO_NLP
Entrenando Logistic Regression...
Entrenando Random Forest...
Entrenando XGBoost...
Entrenando LightGBM...
Escenario: E3_CLINICO_ECG
Entrenando Logistic Regression...
Entrenando Random Forest...
Entrenando XGBoost...
Entrenando LightGBM...
Escenario: E4_CLINICO_NLP_ECG
Entrenando Logistic Regression...
Entrenando Random Forest...
Entrenando XGBoost...
Entrenando LightGBM...


,escenario,modelo,modelo_key,estado,error,n_total,n_train,n_test,features,numeric_features,...,Accuracy,Precision,Recall,F1,ROC_AUC,AUPRC,TN,FP,FN,TP
2,E1_CLINICO,XGBoost,XGBoost,OK,,3779,3023,756,22,20,...,0.984127,0.952381,0.909091,0.930233,0.996734,0.980383,664,4,8,80
3,E1_CLINICO,LightGBM,LightGBM,OK,,3779,3023,756,22,20,...,0.981481,0.930233,0.909091,0.919540,0.995016,0.972917,662,6,8,80
1,E1_CLINICO,Random Forest,RandomForest,OK,,3779,3023,756,22,20,...,0.966931,0.956522,0.750000,0.840764,0.993723,0.961742,665,3,22,66
0,E1_CLINICO,Logistic Regression,LogisticRegression,OK,,3779,3023,756,22,20,...,0.879630,0.490683,0.897727,0.634538,0.930917,0.713767,586,82,9,79
6,E2_CLINICO_NLP,XGBoost,XGBoost,OK,,3779,3023,756,34,32,...,0.985450,0.952941,0.920455,0.936416,0.996683,0.979962,664,4,7,81
7,E2_CLINICO_NLP,LightGBM,LightGBM,OK,,3779,3023,756,34,32,...,0.982804,0.941176,0.909091,0.924855,0.995747,0.975803,663,5,8,80
5,E2_CLINICO_NLP,Random Forest,RandomForest,OK,,3779,3023,756,34,32,...,0.966931,0.970149,0.738636,0.838710,0.993493,0.960587,666,2,23,65
4,E2_CLINICO_NLP,Logistic Regression,LogisticRegression,OK,,3779,3023,756,34,32,...,0.880952,0.493750,0.897727,0.637097,0.930780,0.705131,587,81,9,79
10,E3_CLINICO_ECG,XGBoost,XGBoost,OK,,3779,3023,756,33,31,...,0.988095,0.975904,0.920455,0.947368,0.996768,0.980673,666,2,7,81
11,E3_CLINICO_ECG,LightGBM,LightGBM,OK,,3779,3023,756,33,31,...,0.981481,0.930233,0.909091,0.919540,0.995016,0.972917,662,6,8,80


## 5. Métricas por subconjunto estructural

In [14]:
if subset_metrics_df.empty:
    print("No se generaron métricas por batería: los datasets no contienen columna compatible con subconjunto estructural.")
else:
    display_cols = [
        "escenario", "modelo", "BATERIA", "n_test_bateria", "n_positivos_bateria",
        "n_negativos_bateria", "ROC_AUC", "AUPRC", "F1", "Recall", "Precision"
    ]
    subset_metrics_df.sort_values(["escenario", "modelo", "BATERIA"])[display_cols]


## 6. Exportación de resultados

In [15]:
with pd.ExcelWriter(METRICS_PATH, engine="xlsxwriter") as writer:
    metrics_df.to_excel(writer, index=False, sheet_name="METRICAS")
    summary_df.to_excel(writer, index=False, sheet_name="RESUMEN_ESCENARIOS")
    subset_metrics_df.to_excel(writer, index=False, sheet_name="METRICAS_POR_BATERIA")

with pd.ExcelWriter(METRICS_SUBSETS_PATH, engine="xlsxwriter") as writer:
    subset_metrics_df.to_excel(writer, index=False, sheet_name="METRICAS_POR_BATERIA")

with pd.ExcelWriter(PREDICTIONS_PATH, engine="xlsxwriter") as writer:
    predictions_df.to_excel(writer, index=False, sheet_name="PREDICCIONES")

with pd.ExcelWriter(FEATURES_PATH, engine="xlsxwriter") as writer:
    features_df.to_excel(writer, index=False, sheet_name="VARIABLES")

print("Archivos generados:")
print("-", METRICS_PATH)
print("-", METRICS_SUBSETS_PATH)
print("-", PREDICTIONS_PATH)
print("-", FEATURES_PATH)
print("-", MODEL_DIR)


Archivos generados:
- Metricas_Modelado_TFM.xlsx
- Metricas_Modelado_Subsets_TFM.xlsx
- Predicciones_Modelos_TFM.xlsx
- Variables_Modelado_TFM.xlsx
- modelos_entrenados


## 7. Reporte técnico

In [16]:
lines = []
lines.append("REPORTE MODELADO PREDICTIVO TFM")
lines.append("=" * 70)
lines.append(f"Fecha generacion: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
lines.append(f"Random state: {RANDOM_STATE}")
lines.append(f"Test size: {TEST_SIZE}")
lines.append("")

lines.append("ENTRADAS")
lines.append("-" * 70)
for scenario, filename in SCENARIO_FILES.items():
    lines.append(f"{scenario}: {filename}")
lines.append("")

lines.append("RESUMEN DE ESCENARIOS")
lines.append("-" * 70)
for _, row in summary_df.iterrows():
    lines.append(
        f"{row['escenario']}: registros={row['registros']}, "
        f"features={row['features_entrenamiento']}, "
        f"clase_1={row['clase_1']} ({row['prevalencia_clase_1']:.3f}), "
        f"columna_bateria={row['columna_bateria_detectada']}"
    )
lines.append("")

lines.append("MODELOS EVALUADOS")
lines.append("-" * 70)
for model_key in base_models.keys():
    lines.append(f"- {MODEL_DISPLAY_NAMES.get(model_key, model_key)}")
lines.append("")

lines.append("CONTROL DE LEAKAGE")
lines.append("-" * 70)
lines.append("Los factores utilizados directamente para construir RIESGO_CARDIOMETABOLICO fueron excluidos de los predictores.")
for col in ENDPOINT_FACTORS:
    lines.append(f"- {col}")
lines.append("")

lines.append("MEJORES RESULTADOS GLOBALES POR ESCENARIO SEGUN ROC_AUC")
lines.append("-" * 70)
valid_metrics = metrics_df[metrics_df["estado"] == "OK"].copy()
if len(valid_metrics) == 0:
    lines.append("No se registraron modelos ejecutados correctamente.")
else:
    for scenario, g in valid_metrics.groupby("escenario"):
        best = g.sort_values(["ROC_AUC", "AUPRC", "F1"], ascending=False).iloc[0]
        lines.append(
            f"{scenario}: {best['modelo']} | "
            f"ROC_AUC={best['ROC_AUC']:.4f}, AUPRC={best['AUPRC']:.4f}, F1={best['F1']:.4f}"
        )
lines.append("")

lines.append("METRICAS POR SUBCONJUNTO ESTRUCTURAL")
lines.append("-" * 70)
if subset_metrics_df.empty:
    lines.append("No se generaron metricas por bateria porque no se detecto columna compatible con subconjunto estructural.")
else:
    lines.append(f"Total filas de metricas por bateria: {len(subset_metrics_df)}")
    for scenario, g in subset_metrics_df.groupby("escenario"):
        batteries = sorted([str(x) for x in g["BATERIA"].dropna().unique().tolist()])
        lines.append(f"{scenario}: baterias evaluadas={', '.join(batteries)}")
lines.append("")

lines.append("SALIDAS")
lines.append("-" * 70)
lines.append(str(METRICS_PATH))
lines.append(str(METRICS_SUBSETS_PATH))
lines.append(str(PREDICTIONS_PATH))
lines.append(str(FEATURES_PATH))
lines.append(str(MODEL_DIR))
lines.append("")

lines.append("NOTA METODOLOGICA")
lines.append("-" * 70)
lines.append("Este notebook entrena modelos y calcula metricas base por escenario. La evaluacion incremental entre escenarios se realiza en 07_evaluacion_incremental_ecg.ipynb.")
lines.append("Las metricas por bateria se calculan sobre las predicciones del conjunto de prueba, agrupadas por subconjunto estructural, cuando existe una columna compatible.")

REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")
print(REPORT_PATH.read_text(encoding="utf-8"))


REPORTE MODELADO PREDICTIVO TFM
Fecha generacion: 2026-06-24 15:32:14
Random state: 42
Test size: 0.2

ENTRADAS
----------------------------------------------------------------------
E1_CLINICO: Dataset_E1_Clinico_TFM.xlsx
E2_CLINICO_NLP: Dataset_E2_Clinico_NLP_TFM.xlsx
E3_CLINICO_ECG: Dataset_E3_Clinico_ECG_TFM.xlsx
E4_CLINICO_NLP_ECG: Dataset_E4_Clinico_NLP_ECG_TFM.xlsx

RESUMEN DE ESCENARIOS
----------------------------------------------------------------------
E1_CLINICO: registros=3779, features=22, clase_1=440 (0.116), columna_bateria=SUBSET_BATERIA
E2_CLINICO_NLP: registros=3779, features=34, clase_1=440 (0.116), columna_bateria=SUBSET_BATERIA
E3_CLINICO_ECG: registros=3779, features=33, clase_1=440 (0.116), columna_bateria=SUBSET_BATERIA
E4_CLINICO_NLP_ECG: registros=3779, features=45, clase_1=440 (0.116), columna_bateria=SUBSET_BATERIA

MODELOS EVALUADOS
----------------------------------------------------------------------
- Logistic Regression
- Random Forest
- XGBoost
- Lig

## 8. Consideraciones metodológicas

Este notebook no interpreta el valor incremental del ECG. Su responsabilidad termina en la generación de métricas reproducibles por escenario, modelo y subconjunto estructural cuando corresponde.

La comparación incremental se realiza en `07_evaluacion_incremental_ecg.ipynb` mediante las tablas producidas en este proceso.

Los factores utilizados para construir `RIESGO_CARDIOMETABOLICO` se excluyen de los predictores para evitar aprendizaje directo de la regla de construcción del endpoint.

Las métricas por batería se calculan sobre el conjunto de prueba y permiten evaluar estabilidad del desempeño entre configuraciones estructurales de disponibilidad de información. Cuando un subconjunto contiene una sola clase en el conjunto de prueba, `ROC_AUC` y `AUPRC` se registran como valores no disponibles para evitar estimaciones inválidas.